In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [2]:
from kernels.awa.triton.fwd_kernel import awa_tiled_kernel

In [3]:
import torch
import triton

In [7]:
def run_awa_benchmark_wrapper(q, k, v, meta_tokens, window_size):
    if q.stride(1) < q.stride(2): 
         q = q.transpose(1, 2).contiguous()
         k = k.transpose(1, 2).contiguous()
         v = v.transpose(1, 2).contiguous()
    else:
         q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
         
    B, H, L, D = q.shape
    M_val = meta_tokens.shape[0]
    out = torch.empty_like(q)
    stride_m, stride_d = meta_tokens.stride()

    BLOCK_M = 32
    BLOCK_N = 32
    BLOCK_D = triton.next_power_of_2(D)
    
    next_pow2_M = triton.next_power_of_2(M_val)
    BLOCK_M_META = max(16, next_pow2_M)

    grid = (triton.cdiv(L, BLOCK_M), B * H)
    
    awa_tiled_kernel[grid](
        q, k, v, meta_tokens, out,
        *q.stride(), *k.stride(), *v.stride(),
        0, 0, stride_m, stride_d,
        *out.stride(),
        B, H, L, D, window_size,
        BLOCK_M=BLOCK_M, 
        BLOCK_N=BLOCK_N, 
        BLOCK_D=BLOCK_D,
        BLOCK_M_META=BLOCK_M_META, # Pass the padded size
        M=M_val,                   # Pass the real size for masking
        num_stages=3, num_warps=4 
    )
    
    return out.transpose(1, 2) if q.shape[1] != q.shape[2] else out

In [9]:
# Config
B, L, H, D = 2, 4096, 8, 128
WINDOW_SIZE = 128
NUM_META = 6
dtype = torch.float32
device = "cuda"

print(f"Benchmarking AWA Flash-Triton Kernel...")
print(f"Config: Batch={B}, Len={L}, Heads={H}, Dim={D}, Dtype={dtype}")
print(f"AWA Config: Window={WINDOW_SIZE}, MetaTokens={NUM_META}")

q = torch.randn((B, L, H, D), device=device, dtype=dtype)
k = torch.randn((B, L, H, D), device=device, dtype=dtype)
v = torch.randn((B, L, H, D), device=device, dtype=dtype)

# Meta Tokens
meta_tokens = torch.randn((NUM_META, D), device=device, dtype=dtype)

print("Warming up GPU and compiling kernel...")
for _ in range(10):
    _ = run_awa_benchmark_wrapper(q, k, v, meta_tokens, WINDOW_SIZE)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

print("Starting benchmark loop...")
start_event.record()

n_loops = 100
for _ in range(n_loops):
    _ = run_awa_benchmark_wrapper(q, k, v, meta_tokens, WINDOW_SIZE)

end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops

print(f"Total time ({n_loops} runs): {elapsed_time_ms:.2f} ms")
print(f"Average time per run:      {avg_time:.4f} ms")

Benchmarking AWA Flash-Triton Kernel...
Config: Batch=2, Len=4096, Heads=8, Dim=128, Dtype=torch.float32
AWA Config: Window=128, MetaTokens=6
Warming up GPU and compiling kernel...
Starting benchmark loop...
Total time (100 runs): 62.86 ms
Average time per run:      0.6286 ms
